In [47]:
import pickle
import datetime
import pandas as pd

# Scikit-learn
from sklearn.model_selection import train_test_split as TTS
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

# TensorFlow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy


### Load dataset

In [48]:
data = pd.read_csv('./data/churn_modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [49]:
data['Age'].min(), data['Age'].max()

(np.int64(18), np.int64(92))

In [50]:
data.select_dtypes(include='str').nunique()

Surname      2932
Geography       3
Gender          2
dtype: int64

In [51]:
data.Geography.value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [52]:
data.Gender.value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [53]:
data.Exited.value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [54]:
data.drop(['RowNumber','CustomerId','Surname'], axis=1, inplace=True)

In [55]:
data.sample(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9222,559,France,Male,28,3,141099.43,1,1,1,15607.27,0
2146,850,France,Male,49,5,122486.47,1,0,1,59748.19,0
5791,809,Germany,Female,42,6,64497.94,3,0,1,182436.81,1
4251,601,France,Male,35,2,0.00,2,1,1,118983.18,0
6751,604,France,Female,53,2,121389.78,1,1,1,48201.64,1


### Encode categorical variables

In [56]:
# Label encoding
LE = LabelEncoder()
data.Gender = LE.fit_transform(data.Gender)
data.Gender.value_counts()

Gender
1    5457
0    4543
Name: count, dtype: int64

**Note**:  
In the case of `Geography`, label encoding technique should not be used it can be misunderstood with ordinal values instead nominal values.  
For example:  
`France` = 0  
`Spain` = 1  
`Germany` = 2  
Those can be misunderstood as `France` < `Spain` < `Germany`  
In this case, Onehot encoding technique should be used.

In [57]:
# Onehot encoding
OE = OneHotEncoder(sparse_output=False, drop=None)
geo_encoder = OE.fit_transform(data[['Geography']])


In [58]:
OE.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [59]:
geo_encoded_df = pd.DataFrame(geo_encoder, columns=OE.get_feature_names_out())
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [60]:
# Combine one hot encoded columns with the original data, removing Geography old column
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


### Split dataset into independent and dependent

In [61]:
X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = TTS(X, y, test_size=0.2, random_state=42)
print(X_train.shape)

# Standard scaling to the independent data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

(8000, 12)


In [64]:
# Save encoders and scaler
with open('./preprocessors/classification/gender_label_encoder.pkl', 'wb') as file:
    pickle.dump(LE, file)

with open('./preprocessors/classification/onehot_geo_encoder.pkl', 'wb') as file:
    pickle.dump(OE, file)

with open('./preprocessors/classification/standard_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### ANN Implementation



Our input data is `.csv` file. Our created ANN will be **sequential network** (interconnected).

Our ANN model has the following layers:-
- 01 x Input layer $\Rightarrow$ `X_train.shape[1]` as number of inputs
- 02 x Hidden layers
- 01 x Output layer


**Hidden layer 1**: (2 * 3) + 3b = 6 weights  + 3 bias
**Hidden layer 2**: (3 * 2) + 2b = 6 weights  + 2 bias 
**Output layer**: (2 * 1) + 1b = 2 weights  + 1 bias  

Trainable parameters = (6+3) + (6+2) + (2+1) = 20  

1. Sequential Network
2. Dense for each hidden node
3. Activation function $\rightarrow$ Sigmoid, tanh, ReLU, Leaky ReLU
4. Optimizer $\rightarrow$ Back Propagation $\rightarrow$ Updating Weights
5. Loss function $\downarrow\downarrow\downarrow$
6. Metrics $\rightarrow$ accuracy (MSE, MAE)
7. Training logs $\rightarrow$ Tensorboard logs folder $\rightarrow$ Visualization graph



#### Build ANN model

In [33]:
model = Sequential([Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # HL1 connected with Input Layer
                    Dense(32, activation='relu'),   # HL2
                    Dense(1, activation='sigmoid')])  # Output Layer

/home/htet-aung-lynn/Study/E2E-Churn-Prediction-with-ANN/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
optmr = Adam(learning_rate=0.01)
loss = BinaryCrossentropy()

In [36]:
# Complie model
model.compile(optimizer = "adam",               # 
              loss = "binary_crossentropy",     # 
              metrics = ["accuracy"])            # Get as a list

`model.compile(optimizer="adam",...)` is a fixed learning rate. We want to use our own learning rate.

In [37]:
# Set up Tensorboard
log_dir = "logs/classification/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [38]:
tensorflow_callback = TensorBoard(log_dir=log_dir)

`TensorBoard()` is a visualization and monitoring tool for TensorFlow/Keras training.  

`TensorBoard()` can visualize things such as:

- Training loss
- Validation loss
- Training accuracy
- Validation accuracy
- Learning rate
- Model graph
- Histograms and other training information

In [39]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor = 'val_loss', 
                                        patience = 10,  # Allow up to 10 consecutive epochs without improvement before stopping training
                                        restore_best_weights = True) # restore the best weight after monitoring next 10 epochs

`EarlyStopping()` prevents the model from continuing to train when its validation performance stops improving.

In [40]:
history = model.fit(X_train_scaled,
                    y_train,
                    # validation_split = 0.2,
                    validation_data = (X_test, y_test),
                    epochs = 100,
                    # batch_size = 32,
                    callbacks = [tensorflow_callback, early_stopping_callback])

Epoch 1/100


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7872 - loss: 0.4723 - val_accuracy: 0.8035 - val_loss: 7370.2422
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 745us/step - accuracy: 0.8396 - loss: 0.3905 - val_accuracy: 0.8035 - val_loss: 6452.0786
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 920us/step - accuracy: 0.8526 - loss: 0.3577 - val_accuracy: 0.8035 - val_loss: 6240.7231
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 981us/step - accuracy: 0.8568 - loss: 0.3465 - val_accuracy: 0.8035 - val_loss: 3722.3140
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 895us/step - accuracy: 0.8590 - loss: 0.3405 - val_accuracy: 0.8035 - val_loss: 4546.1001
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 936us/step - accuracy: 0.8589 - loss: 0.3366 - val_accuracy: 0.5250 - val_loss: 2578.2732
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 852us/step - accuracy: 0.8609 - loss: 0.3341 - val_accuracy: 0.4800 - val_loss: 4258.9180
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 949us/step - accuracy: 0.8635 - 

In [41]:
# Save model.h5 -> h5 is compatible for Keras
model.save('./dl_model/classification_model.h5')

In [42]:
#  Load Tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [43]:
import tensorboard
print(tensorboard.__version__)

2.21.0


In [46]:
%tensorboard --logdir logs/classification/

Reusing TensorBoard on port 6007 (pid 487738), started 0:00:13 ago. (Use '!kill 487738' to kill it.)